# Лабораторная работа 4

Студент: Захаров Никита Константинович
Группа: 6403-010302D


Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

gpus = tf.config.list_physical_devices('GPU')
device = '/GPU:0' if gpus else '/CPU:0'

%matplotlib inline


I0000 00:00:1777626529.806954   29448 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777626529.868702   29448 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777626532.054625   29448 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1777626534.031383   29448 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries men

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

/home/ZakharovNK/.local/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.conv1 = tf.keras.layers.Conv2D(channel_1, 5, padding='same', activation='relu', kernel_initializer=initializer)
        self.conv2 = tf.keras.layers.Conv2D(channel_2, 3, padding='same', activation='relu', kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax', kernel_initializer=initializer)
        
    def call(self, x, training=False):
        if x.shape.rank == 4 and x.shape[1] == 3 and x.shape[-1] != 3:
            x = tf.transpose(x, [0, 2, 3, 1])
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)
        return scores


In [7]:
def test_ThreeLayerConvNet():    
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [8]:
print_every = 100

def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    with tf.device(device):
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        steps_per_epoch = (train_dset.X.shape[0] + train_dset.batch_size - 1) // train_dset.batch_size
        total_steps = steps_per_epoch * num_epochs
        t = 0
        for epoch in range(num_epochs):
            train_loss.reset_state()
            train_accuracy.reset_state()
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
                gradients = tape.gradient(loss, model.trainable_variables)
                optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                train_loss.update_state(loss)
                train_accuracy.update_state(y_np, scores)
                if t % print_every == 0 or t == total_steps - 1:
                    val_loss.reset_state()
                    val_accuracy.reset_state()
                    for test_x, test_y in val_dset:
                        prediction = model(test_x, training=False)
                        t_loss = loss_fn(test_y, prediction)
                        val_loss.update_state(t_loss)
                        val_accuracy.update_state(test_y, prediction)
                    print(
                        f'step {t + 1}/{total_steps} | epoch {epoch + 1}/{num_epochs} | '
                        f'train_loss {train_loss.result().numpy():.4f} | '
                        f'train_acc {train_accuracy.result().numpy() * 100:.2f}% | '
                        f'val_loss {val_loss.result().numpy():.4f} | '
                        f'val_acc {val_accuracy.result().numpy() * 100:.2f}%'
                    )
                t += 1


In [9]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

step 1/766 | epoch 1/1 | train_loss 3.1014 | train_acc 12.50% | val_loss 2.9257 | val_acc 13.20%
step 101/766 | epoch 1/1 | train_loss 2.2616 | train_acc 28.31% | val_loss 1.8750 | val_acc 38.00%
step 201/766 | epoch 1/1 | train_loss 2.0845 | train_acc 32.03% | val_loss 1.7985 | val_acc 39.00%
step 301/766 | epoch 1/1 | train_loss 2.0100 | train_acc 33.69% | val_loss 1.8592 | val_acc 38.30%
step 401/766 | epoch 1/1 | train_loss 1.9384 | train_acc 35.66% | val_loss 1.7024 | val_acc 41.60%
step 501/766 | epoch 1/1 | train_loss 1.8908 | train_acc 36.88% | val_loss 1.6437 | val_acc 42.80%
step 601/766 | epoch 1/1 | train_loss 1.8601 | train_acc 37.83% | val_loss 1.6679 | val_acc 41.80%
step 701/766 | epoch 1/1 | train_loss 1.8333 | train_acc 38.61% | val_loss 1.6091 | val_acc 44.10%
step 766/766 | epoch 1/1 | train_loss 1.8169 | train_acc 38.99% | val_loss 1.5868 | val_acc 46.20%


Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [10]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    return model

def optimizer_init_fn():
    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)


step 1/766 | epoch 1/1 | train_loss 2.9607 | train_acc 4.69% | val_loss 5.6781 | val_acc 9.50%
step 101/766 | epoch 1/1 | train_loss 2.0986 | train_acc 28.47% | val_loss 1.7618 | val_acc 42.20%
step 201/766 | epoch 1/1 | train_loss 1.8840 | train_acc 34.92% | val_loss 1.5784 | val_acc 45.30%
step 301/766 | epoch 1/1 | train_loss 1.7716 | train_acc 38.30% | val_loss 1.5050 | val_acc 47.30%
step 401/766 | epoch 1/1 | train_loss 1.6847 | train_acc 41.05% | val_loss 1.4223 | val_acc 49.10%
step 501/766 | epoch 1/1 | train_loss 1.6246 | train_acc 43.00% | val_loss 1.3712 | val_acc 51.50%
step 601/766 | epoch 1/1 | train_loss 1.5833 | train_acc 44.41% | val_loss 1.3458 | val_acc 53.20%
step 701/766 | epoch 1/1 | train_loss 1.5490 | train_acc 45.57% | val_loss 1.3216 | val_acc 53.40%
step 766/766 | epoch 1/1 | train_loss 1.5274 | train_acc 46.32% | val_loss 1.2848 | val_acc 53.90%


# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [11]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax', 
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate) 

train_part34(model_init_fn, optimizer_init_fn)

/home/ZakharovNK/.local/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


step 1/766 | epoch 1/1 | train_loss 2.7578 | train_acc 14.06% | val_loss 2.6346 | val_acc 13.10%
step 101/766 | epoch 1/1 | train_loss 2.2303 | train_acc 28.28% | val_loss 1.8372 | val_acc 40.40%
step 201/766 | epoch 1/1 | train_loss 2.0670 | train_acc 32.41% | val_loss 1.8060 | val_acc 41.30%
step 301/766 | epoch 1/1 | train_loss 1.9928 | train_acc 34.23% | val_loss 1.8560 | val_acc 37.80%
step 401/766 | epoch 1/1 | train_loss 1.9238 | train_acc 36.08% | val_loss 1.7165 | val_acc 41.00%
step 501/766 | epoch 1/1 | train_loss 1.8793 | train_acc 37.21% | val_loss 1.6669 | val_acc 43.40%
step 601/766 | epoch 1/1 | train_loss 1.8509 | train_acc 38.11% | val_loss 1.6593 | val_acc 43.40%
step 701/766 | epoch 1/1 | train_loss 1.8271 | train_acc 38.71% | val_loss 1.6147 | val_acc 44.90%
step 766/766 | epoch 1/1 | train_loss 1.8107 | train_acc 39.13% | val_loss 1.5888 | val_acc 46.90%


Альтернативный менее гибкий способ обучения:

In [12]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - loss: 1.8158 - sparse_categorical_accuracy: 0.3919 - val_loss: 1.7426 - val_sparse_categorical_accuracy: 0.4210
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 1.6889 - sparse_categorical_accuracy: 0.4280


[1.6889312267303467, 0.42800000309944153]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [13]:
def model_init_fn():
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, 5, padding='same', activation='relu', kernel_initializer=initializer),
        tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu', kernel_initializer=initializer),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer),
    ])
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)


/home/ZakharovNK/.local/lib/python3.12/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


step 1/766 | epoch 1/1 | train_loss 2.9418 | train_acc 3.12% | val_loss 3.5762 | val_acc 10.90%
step 101/766 | epoch 1/1 | train_loss 1.9419 | train_acc 32.67% | val_loss 1.6694 | val_acc 42.50%
step 201/766 | epoch 1/1 | train_loss 1.7802 | train_acc 37.86% | val_loss 1.5290 | val_acc 46.30%
step 301/766 | epoch 1/1 | train_loss 1.6891 | train_acc 40.78% | val_loss 1.4830 | val_acc 46.60%
step 401/766 | epoch 1/1 | train_loss 1.6158 | train_acc 43.17% | val_loss 1.3996 | val_acc 52.00%
step 501/766 | epoch 1/1 | train_loss 1.5652 | train_acc 44.88% | val_loss 1.3520 | val_acc 51.80%
step 601/766 | epoch 1/1 | train_loss 1.5309 | train_acc 46.19% | val_loss 1.3211 | val_acc 54.30%
step 701/766 | epoch 1/1 | train_loss 1.5001 | train_acc 47.31% | val_loss 1.2950 | val_acc 53.00%
step 766/766 | epoch 1/1 | train_loss 1.4792 | train_acc 48.01% | val_loss 1.2645 | val_acc 57.90%


In [14]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 1.6021 - sparse_categorical_accuracy: 0.4375 - val_loss: 1.3712 - val_sparse_categorical_accuracy: 0.5290
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.3626 - sparse_categorical_accuracy: 0.5141


[1.3626320362091064, 0.5141000151634216]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [15]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [16]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

step 1/766 | epoch 1/1 | train_loss 3.2534 | train_acc 12.50% | val_loss 2.8746 | val_acc 11.70%
step 101/766 | epoch 1/1 | train_loss 2.2589 | train_acc 28.50% | val_loss 1.8704 | val_acc 38.80%
step 201/766 | epoch 1/1 | train_loss 2.0938 | train_acc 32.06% | val_loss 1.8232 | val_acc 40.70%
step 301/766 | epoch 1/1 | train_loss 2.0159 | train_acc 34.14% | val_loss 1.8616 | val_acc 37.10%
step 401/766 | epoch 1/1 | train_loss 1.9437 | train_acc 36.06% | val_loss 1.7122 | val_acc 42.10%
step 501/766 | epoch 1/1 | train_loss 1.8975 | train_acc 37.10% | val_loss 1.6714 | val_acc 43.20%
step 601/766 | epoch 1/1 | train_loss 1.8661 | train_acc 37.90% | val_loss 1.6833 | val_acc 42.60%
step 701/766 | epoch 1/1 | train_loss 1.8384 | train_acc 38.60% | val_loss 1.6429 | val_acc 44.50%
step 766/766 | epoch 1/1 | train_loss 1.8227 | train_acc 38.93% | val_loss 1.5934 | val_acc 45.60%


Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [ ]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        initializer = tf.initializers.HeNormal()
        self.conv1 = tf.keras.layers.Conv2D(64, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPool2D()
        self.drop1 = tf.keras.layers.Dropout(0.25)
        self.conv3 = tf.keras.layers.Conv2D(128, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.conv4 = tf.keras.layers.Conv2D(128, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPool2D()
        self.drop2 = tf.keras.layers.Dropout(0.35)
        self.conv5 = tf.keras.layers.Conv2D(256, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn5 = tf.keras.layers.BatchNormalization()
        self.gap = tf.keras.layers.GlobalAveragePooling2D()
        self.fc1 = tf.keras.layers.Dense(256, activation='relu', kernel_initializer=initializer)
        self.drop3 = tf.keras.layers.Dropout(0.5)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer)
    
    def call(self, input_tensor, training=False):
        x = input_tensor
        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        x = self.drop1(x, training=training)
        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)
        x = self.drop2(x, training=training)
        x = self.conv5(x)
        x = self.bn5(x, training=training)
        x = tf.nn.relu(x)
        x = self.gap(x)
        x = self.fc1(x)
        x = self.drop3(x, training=training)
        x = self.fc2(x)
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate) 

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)


step 1/7660 | epoch 1/10 | train_loss 2.6120 | train_acc 10.94% | val_loss 4.3790 | val_acc 13.60%
step 701/7660 | epoch 1/10 | train_loss 1.4456 | train_acc 47.28% | val_loss 1.3254 | val_acc 52.90%
step 1401/7660 | epoch 2/10 | train_loss 1.0719 | train_acc 61.74% | val_loss 0.9393 | val_acc 65.60%
step 2101/7660 | epoch 3/10 | train_loss 0.9296 | train_acc 67.31% | val_loss 0.8465 | val_acc 72.30%
step 2801/7660 | epoch 4/10 | train_loss 0.8338 | train_acc 70.62% | val_loss 1.1297 | val_acc 62.00%
step 3501/7660 | epoch 5/10 | train_loss 0.7520 | train_acc 73.87% | val_loss 0.8316 | val_acc 71.90%
step 4201/7660 | epoch 6/10 | train_loss 0.6788 | train_acc 76.26% | val_loss 0.7345 | val_acc 76.50%
step 4901/7660 | epoch 7/10 | train_loss 0.6325 | train_acc 78.04% | val_loss 0.7635 | val_acc 74.60%
step 5601/7660 | epoch 8/10 | train_loss 0.5825 | train_acc 79.57% | val_loss 0.7800 | val_acc 74.70%
step 6301/7660 | epoch 9/10 | train_loss 0.5400 | train_acc 81.41% | val_loss 0.6250 |

Опишите все эксперименты, результаты. Сделайте выводы.

Сначала была обучена двухслойная полносвязная сеть через Model Subclassing API: validation accuracy выросла примерно до 0.46. Затем были проверены эквивалентные реализации через Sequential и Functional API, которые дали очень близкий результат на той же архитектуре. Базовая трёхслойная CNN оказалась заметно сильнее полносвязных моделей и подняла validation accuracy примерно до 0.54-0.58. После этого архитектура была усложнена: добавлены дополнительные сверточные блоки, BatchNormalization, MaxPooling, Dropout и GlobalAveragePooling. Лучшая модель с блоками 64-64-128-128-256 и оптимизатором Adam с learning_rate=1e-3 превысила требуемый порог 70% уже к 3-й эпохе и завершила обучение с validation accuracy около 0.819 на 10-й эпохе. Вывод: для CIFAR-10 основной прирост качества дали увеличение глубины сверточной сети, нормализация и регуляризация, а требование задания по достижению не менее 70% accuracy на валидации выполнено с хорошим запасом.
